# Convert daily transpiration, evaporation, irrigation, and precipitation to hourly values

FAO-56 calculations are all performed at daily time step. Re-scale daily transpiration and evaporation to hourly values, using hourly reference ET as a scaling factor for transpiration and hourly daylight as a scaling factor for evaporation.

In [ ]:
import s3fs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Set up S3 access
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

# Load in hourly reference ET and precipitation for each site
ref_et = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/ref_et.csv', index_col=0, parse_dates=True)
precip = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/precip.csv', index_col=0, parse_dates=True)
rn = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/rn.csv', index_col=0, parse_dates=True)

all_sites = ref_et.columns

# Load in daily FAO-56 data
all_fao_output = {}
for site in tqdm(all_sites, desc='Iterating over sites...'):
    fao_output = pd.read_csv(f'{s3_base_path}/input-data/processed-data/climate/{site}_FAOWaterBalance.csv', index_col=0, parse_dates=True)
    all_fao_output[site] = fao_output

In [ ]:
# Caclulate hourly weights for evaporation and transpiration
# Daily ETref from hourly forcing
daily_ref_et = ref_et.resample("D").sum(min_count=24)

# Hourly transpiration weights within each day: hourly ETref / daily ETref
# Days with zero daily ETref get zero weights
daily_ref_et_expanded = daily_ref_et.reindex(ref_et.index.normalize())
daily_ref_et_expanded.index = ref_et.index
hourly_weights_transpiration = ref_et.div(daily_ref_et_expanded)

# Double check weights
daily_weights = hourly_weights_transpiration.resample('D').sum()
mask = ~np.isclose(daily_weights.values, 1, atol=0.01)
if mask.sum() > 0:
    print('Daily transpiration weights that don\'t sum to 1:')
    for i, site in enumerate(daily_weights.columns):
        if mask[:, i].sum() > 0:
            print(f'{site+':':<15} {mask[:, i].sum()}')

rn = rn.clip(0)
daily_rn = rn.resample("D").sum(min_count=24)
daily_rn_expanded = daily_rn.reindex(rn.index.normalize())
daily_rn_expanded.index = rn.index
hourly_weights_evap = rn.div(daily_rn_expanded)

# Double check weights
daily_weights = hourly_weights_evap.resample('D').sum()
mask = ~np.isclose(daily_weights.values, 1, atol=0.01)
if mask.sum() > 0:
    print('Daily evaporation weights that don\'t sum to 1:')
    for i, site in enumerate(daily_weights.columns):
        if mask[:, i].sum() > 0:
            print(f'{site+':':<15} {mask[:, i].sum()}')

In [ ]:
# There are a few days for Palouse in which hourly Rn is always less than 0
# For these days, spread out bare soil evaporation evenly across 24 hours
negative_rn_mask = daily_weights['Palouse'] == 0
for idx in daily_weights.index[negative_rn_mask]:
    datestr = idx.strftime('%Y-%m-%d')
    hourly_weights_evap.loc[f'{datestr} 00:00':f'{datestr} 23:00', 'Palouse'] = 1/24

# Double weights again
daily_weights = hourly_weights_evap.resample('D').sum()
mask = ~np.isclose(daily_weights.values, 1, atol=0.01)
print('Daily evaporation weights that don\'t sum to 1:')
if mask.sum() > 0:
    for i, site in enumerate(daily_weights.columns):
        if mask[:, i].sum() > 0:
            print(f'{site+':':<15} {mask[:, i].sum()}')

In [ ]:
# Initialize output DataFrames
hourly_T = pd.DataFrame(index=ref_et.index, columns=all_sites, dtype=float)
hourly_E = pd.DataFrame(index=ref_et.index, columns=all_sites, dtype=float)

# Calculate hourly irrigation and net surface water input (P + I - E)
irrig_hours = np.arange(6, 12)  # Run irrigation from 6AM to 12PM
n_irrig_hours = len(irrig_hours)
irrig_window_mask = ref_et.index.hour.isin(irrig_hours)

hourly_irrig = pd.DataFrame(index=ref_et.index, columns=all_sites, dtype=float)
hourly_surface_water = pd.DataFrame(index=ref_et.index, columns=all_sites, dtype=float)

for site in all_sites:
    fao_daily = all_fao_output[site].copy()
    fao_daily.index = pd.to_datetime(fao_daily.index).normalize()

    # Expand daily values to hourly timestamps
    daily_T_expanded = fao_daily["T"].reindex(ref_et.index.normalize())
    daily_E_expanded = fao_daily["E"].reindex(ref_et.index.normalize())

    daily_T_expanded.index = ref_et.index
    daily_E_expanded.index = ref_et.index

    # Redistribute daily T and E according to hourly ETref fraction
    hourly_T[site] = daily_T_expanded * hourly_weights_transpiration[site]
    hourly_E[site] = daily_E_expanded * hourly_weights_evap[site]

    # Irrigation is applied uniformly from 6am to 12pm
    daily_I_expanded = fao_daily["Irrig"].reindex(ref_et.index.normalize())
    daily_I_expanded.index = ref_et.index

    hourly_irrig[site] = 0.0
    hourly_irrig.loc[irrig_window_mask, site] = daily_I_expanded.loc[irrig_window_mask] / n_irrig_hours

    # Calculate hourly surface water balance as P + I - E
    hourly_surface_water[site] = precip[site] + hourly_irrig[site] - hourly_E[site]
    hourly_surface_water[f'{site}_precip_mm.hr'] = precip[site]
    hourly_surface_water[f'{site}_irrig_mm.hr'] = hourly_irrig[site]
    hourly_surface_water[f'{site}_evap_mm.hr'] = hourly_E[site]

    forcing = pd.DataFrame(columns=['transpiration_mm.hr', 'surface_flux_mm.hr'], index=ref_et.index)
    forcing['transpiration_mm.hr'] = hourly_T[site]
    forcing['surface_flux_mm.hr'] = hourly_surface_water[site]

    forcing.to_csv(f'{s3_base_path}/input-data/processed-data/climate/{site}_hourly_forcing.csv')

cols_to_save = [c for c in hourly_surface_water if c not in all_sites]
hourly_surface_water[cols_to_save].to_parquet(f'{s3_base_path}/input-data/processed-data/climate/HourlySurfaceWaterBalance.parquet')

## Checks and plots

In [ ]:
# Check daily mass balance after hourly redistribution
daily_check_records = []

for site in all_sites:
    fao_daily = all_fao_output[site].copy()
    fao_daily.index = pd.to_datetime(fao_daily.index).normalize()

    T_daily_from_hourly = hourly_T[site].resample("D").sum()
    E_daily_from_hourly = hourly_E[site].resample("D").sum()

    common_days = fao_daily.index.intersection(T_daily_from_hourly.index)

    tmp = pd.DataFrame({
        "site": site,
        "T_error_mm": T_daily_from_hourly.loc[common_days].values - pd.to_numeric(fao_daily.loc[common_days, "T"], errors="coerce").values,
        "E_error_mm": E_daily_from_hourly.loc[common_days].values - pd.to_numeric(fao_daily.loc[common_days, "E"], errors="coerce").values,
    })

    daily_check_records.append(tmp)

daily_check = pd.concat(daily_check_records, ignore_index=True)

daily_check[["T_error_mm", "E_error_mm"]].describe()

In [ ]:
# Ensure daily irrigation equals sum of hourly irrigation
site = all_sites[0]

daily_irrig_from_hourly = hourly_irrig[site].resample("D").sum()
daily_irrig_from_fao = all_fao_output[site]["Irrig"].copy()
daily_irrig_from_fao.index = pd.to_datetime(daily_irrig_from_fao.index).normalize()
daily_irrig_from_fao = pd.to_numeric(daily_irrig_from_fao, errors="coerce")

irrig_error = (
    daily_irrig_from_hourly
    .reindex(daily_irrig_from_fao.index)
    .sub(daily_irrig_from_fao)
)

print(f'Largest daily irrigation difference: {irrig_error.abs().max():.2e} mm')

In [ ]:
# Check mean diurnal cycle of ETref, T, E, and Irrig
site = np.random.choice(all_sites, 1)[0]

diurnal_df = pd.DataFrame({
    "ETref": ref_et[site],
    "T": hourly_T[site],
    "E": hourly_E[site],
    'I': hourly_irrig[site],
    "P": precip[site],
}).dropna()

# Restrict to growing season months to make transpiration pattern clear
growing_months = [5, 6, 7, 8, 9, 10]
diurnal_growing = diurnal_df.loc[diurnal_df.index.month.isin(growing_months)]

diurnal_mean = diurnal_growing.groupby(diurnal_growing.index.hour).mean()

fig, ax = plt.subplots(figsize=(8, 4))

diurnal_mean[["ETref", "T", "E", "I", "P"]].plot(ax=ax, marker="o")

ax.set(xlabel='Hour of day', ylabel="Mean hourly flux, mm/hr", title=f"{site}: Mean hourly values during growing season")

In [ ]:
# Plot hourly surface water balance (P + I - E) for a random site
site = np.random.choice(all_sites, 1)[0]

fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(hourly_surface_water[site], bins=50, edgecolor='k', color='0.5', alpha=0.7)
ax.set(xlabel='Hourly surface water balance (P + I - E), mm/hr', ylabel='Frequency', title=f'{site}: Distribution of hourly surface water balance',
       yscale='log')